# APMC Data Transforms (Lite)

A streamlined version of the transforms notebook covering basic cleaning through family text rollup.
Only five source columns are retained from the original 62: `nRawStateDataID`, `nRawStateDataID_parent`,
`nStateID`, `txtRawStateData`, and `txtDefaultImage`. All computed columns stay — no cleanup drops at the end.

## Table of Contents

- [0. Setup & Data Loading](#section-0)
- [1. Column Reduction](#section-1)
- [2. Replace NULL/N-A/-- Strings with Actual Nulls](#section-2)
- [3. Whitespace Normalization](#section-3)
- [4. Garbage Record Detection](#section-4)
- [5. Parent-Child Relationship Repair](#section-5)
  - [5.1 Type B: Duplicate Records from Botched Import](#section-5-1)
  - [5.2 Parent-Child Reassignment](#section-5-2)
- [6. Family Text Rollup](#section-6)
- [7. Default / Main Image Synchronization](#section-7)
- [8. State Lookup Merge](#section-8)
- [9. Save Results](#section-9)

---

<a id="section-0"></a>

## 0. Setup & Data Loading

In [27]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# Configuration
INPUT_FILE = './wip/in/tblRawStateData_orig.csv'
OUTPUT_FILE = './wip/out/postmark.csv'
GARBAGE_FILE = './wip/out/tblRawStateData_garbage.csv'
DUPLICATES_FILE = './wip/out/tblRawStateData_duplicates.csv'

print(f"Input file: {INPUT_FILE}")
print(f"Cleaned output: {OUTPUT_FILE}")
print(f"Garbage records: {GARBAGE_FILE}")
print(f"Duplicate records: {DUPLICATES_FILE}")

Input file: ./wip/in/tblRawStateData_orig.csv
Cleaned output: ./wip/out/postmark.csv
Garbage records: ./wip/out/tblRawStateData_garbage.csv
Duplicate records: ./wip/out/tblRawStateData_duplicates.csv


In [28]:
# Load the CSV with all columns as strings initially to preserve data
df = pd.read_csv(INPUT_FILE, dtype=str, low_memory=False, keep_default_na=False)

original_count = len(df)
print(f"Loaded {original_count:,} records with {len(df.columns)} columns")

Loaded 52,046 records with 62 columns


---

<a id="section-1"></a>

## 1. Column Reduction

Keep only the five columns that matter: the record ID, its parent pointer,
the state, the authoritative catalog text, and the default image filename
(needed for image table synchronization). Everything else is either a parsed
derivative, a workflow artifact, or an implementation concern.

In [29]:
KEEP_COLUMNS = [
    'nRawStateDataID',
    'nRawStateDataID_parent',
    'nStateID',
    'txtRawStateData',
    'txtDefaultImage',
]

missing = [c for c in KEEP_COLUMNS if c not in df.columns]
if missing:
    raise ValueError(f"Expected columns not found: {missing}")

dropped = [c for c in df.columns if c not in KEEP_COLUMNS]
df = df[KEEP_COLUMNS]

print(f"Dropped {len(dropped)} columns, kept {len(df.columns)}:")
for col in df.columns:
    print(f"  - {col}")

Dropped 57 columns, kept 5:
  - nRawStateDataID
  - nRawStateDataID_parent
  - nStateID
  - txtRawStateData
  - txtDefaultImage


---

<a id="section-2"></a>

## 2. Replace NULL, N/A, and '--' Strings with Actual Nulls

The data contains widespread pollution from literal string values
"NULL", "n/a", "NA", "--" that should be actual null values.

In [30]:
def count_null_strings(dataframe, patterns):
    """Count occurrences of NULL-like strings across all columns."""
    counts = {}
    for col in dataframe.columns:
        col_lower = dataframe[col].astype(str).str.strip().str.lower()
        pattern_set = {p.lower() for p in patterns}
        count = col_lower.isin(pattern_set).sum()
        if count > 0:
            counts[col] = count
    return counts

null_patterns = ['NULL', 'n/a', 'N/A', 'na', 'NA', '--']

before_counts = count_null_strings(df, null_patterns)
total_before = sum(before_counts.values())

print(f"NULL/N-A/-- strings found BEFORE replacement: {total_before:,}")
for col, count in sorted(before_counts.items(), key=lambda x: -x[1]):
    print(f"  {col}: {count:,}")

NULL/N-A/-- strings found BEFORE replacement: 46,516
  txtDefaultImage: 45,408
  txtRawStateData: 1,108


In [31]:
def replace_null_strings(dataframe, patterns):
    """Replace NULL-like string patterns with actual NaN values."""
    df_cleaned = dataframe.copy()
    pattern_regex = '|'.join([f'^{re.escape(p)}$' for p in patterns])

    for col in df_cleaned.columns:
        df_cleaned[col] = df_cleaned[col].replace(
            to_replace=pattern_regex,
            value=np.nan,
            regex=True
        )
        # Also handle after stripping whitespace
        mask = df_cleaned[col].astype(str).str.strip().str.lower().isin(
            [p.lower() for p in patterns]
        )
        df_cleaned.loc[mask, col] = np.nan

    return df_cleaned

df = replace_null_strings(df, null_patterns)

after_counts = count_null_strings(df, null_patterns)
total_after = sum(after_counts.values()) if after_counts else 0

print(f"NULL/N-A strings AFTER replacement: {total_after:,}")
print(f"Strings converted to null: {total_before - total_after:,}")

NULL/N-A strings AFTER replacement: 0
Strings converted to null: 46,516


---

<a id="section-3"></a>

## 3. Whitespace Normalization

Removes leading/trailing whitespace, collapses multiple spaces,
and normalizes newlines.

In [32]:
def count_whitespace_issues(dataframe):
    """Count whitespace issues across string columns."""
    issues = {'leading_trailing': 0, 'multiple_inner': 0}

    for col in dataframe.columns:
        col_series = dataframe[col].astype(str)
        stripped = col_series.str.strip()
        issues['leading_trailing'] += (col_series != stripped).sum()
        issues['multiple_inner'] += col_series.str.contains(
            r'  +', regex=True, na=False
        ).sum()

    return issues

ws_before = count_whitespace_issues(df)
print(f"Whitespace issues BEFORE normalization:")
print(f"  Leading/trailing whitespace: {ws_before['leading_trailing']:,}")
print(f"  Multiple inner spaces: {ws_before['multiple_inner']:,}")

Whitespace issues BEFORE normalization:
  Leading/trailing whitespace: 50,664
  Multiple inner spaces: 129


In [33]:
def normalize_whitespace(dataframe):
    """Normalize whitespace in all string columns."""
    df_cleaned = dataframe.copy()

    for col in df_cleaned.columns:
        if df_cleaned[col].dtype == object:
            df_cleaned[col] = df_cleaned[col].astype(str)
            df_cleaned[col] = df_cleaned[col].str.replace(
                r'[\r\n]+', ' ', regex=True
            )
            df_cleaned[col] = df_cleaned[col].str.strip()
            df_cleaned[col] = df_cleaned[col].str.replace(
                r'  +', ' ', regex=True
            )
            # Restore NaN for 'nan' strings created by astype
            df_cleaned.loc[df_cleaned[col] == 'nan', col] = np.nan

    return df_cleaned

df = normalize_whitespace(df)

ws_after = count_whitespace_issues(df)
print(f"Whitespace issues AFTER normalization:")
print(f"  Leading/trailing whitespace: {ws_after['leading_trailing']:,}")
print(f"  Multiple inner spaces: {ws_after['multiple_inner']:,}")

Whitespace issues AFTER normalization:
  Leading/trailing whitespace: 0
  Multiple inner spaces: 0


---

<a id="section-4"></a>

## 4. Garbage Record Detection

A record is garbage if ALL three criteria are met:

1. **Childless** -- not referenced as any other record's `nRawStateDataID_parent`
2. **Orphaned** -- parent is 0, empty, or self-referencing
3. **Empty `txtRawStateData`** -- no catalog text at all

No workflow flags. If it has no text, no children, and no real parent, it's junk.

In [34]:
# Criterion 1: Childless (not used as parent by other records)
used_as_parent = set(df['nRawStateDataID_parent'].dropna().astype(str).unique())
df['_is_childless'] = ~df['nRawStateDataID'].astype(str).isin(used_as_parent)
print(f"Unique parent IDs referenced: {len(used_as_parent):,}")
print(f"1. Childless records: {df['_is_childless'].sum():,}")

# Criterion 2: Orphaned (parent is 0, empty, or self)
def is_orphaned(row):
    parent = str(row['nRawStateDataID_parent']) if pd.notna(row['nRawStateDataID_parent']) else ''
    self_id = str(row['nRawStateDataID']) if pd.notna(row['nRawStateDataID']) else ''
    return parent == '' or parent == '0' or parent == self_id

df['_is_orphaned'] = df.apply(is_orphaned, axis=1)
print(f"2. Orphaned records (parent=0/empty/self): {df['_is_orphaned'].sum():,}")

# Criterion 3: Empty txtRawStateData
df['_empty_content'] = (
    df['txtRawStateData'].isna()
    | (df['txtRawStateData'].astype(str).str.strip() == '')
)
print(f"3. Empty txtRawStateData: {df['_empty_content'].sum():,}")

Unique parent IDs referenced: 43,055
1. Childless records: 8,992
2. Orphaned records (parent=0/empty/self): 51,842
3. Empty txtRawStateData: 1,119


In [35]:
# A record is garbage only if ALL conditions are true
df['_is_garbage'] = (
    df['_is_childless']
    & df['_is_orphaned']
    & df['_empty_content']
)

garbage_count = df['_is_garbage'].sum()
clean_count = len(df) - garbage_count

print(f"\n=== GARBAGE DETECTION RESULTS ===")
print(f"Total records: {len(df):,}")
print(f"Garbage records (ALL criteria met): {garbage_count:,} ({garbage_count/len(df)*100:.2f}%)")
print(f"Clean records: {clean_count:,} ({clean_count/len(df)*100:.2f}%)")


=== GARBAGE DETECTION RESULTS ===
Total records: 52,046
Garbage records (ALL criteria met): 1,094 (2.10%)
Clean records: 50,952 (97.90%)


In [36]:
# Progressive criteria analysis
c1 = df['_is_childless']
c2 = df['_is_orphaned']
c3 = df['_empty_content']

print(f"Starting records: {len(df):,}")
print(f"After criterion 1 (childless): {c1.sum():,}")
print(f"After criteria 1+2 (+ orphaned): {(c1 & c2).sum():,}")
print(f"After criteria 1+2+3 (+ empty content): {(c1 & c2 & c3).sum():,}")

Starting records: 52,046
After criterion 1 (childless): 8,992
After criteria 1+2 (+ orphaned): 8,788
After criteria 1+2+3 (+ empty content): 1,094


In [37]:
# Show sample garbage records
if garbage_count > 0:
    print("=== Sample Garbage Records (first 10) ===")
    garbage_df = df[df['_is_garbage']]
    print(garbage_df.head(10).to_string())
else:
    print("No garbage records detected.")

=== Sample Garbage Records (first 10) ===
     nRawStateDataID nRawStateDataID_parent nStateID txtRawStateData txtDefaultImage  _is_childless  _is_orphaned  _empty_content  _is_garbage
7233            7235                      0        7             NaN             NaN           True          True            True         True
7234            7236                      0        7             NaN             NaN           True          True            True         True
7235            7237                      0        7             NaN             NaN           True          True            True         True
7236            7238                      0        7             NaN             NaN           True          True            True         True
7237            7239                      0        7             NaN             NaN           True          True            True         True
7238            7240                      0        7             NaN             NaN           True 

---

<a id="section-5"></a>

## 5. Parent-Child Relationship Repair

The `nRawStateDataID_parent` field denotes hierarchical relationships where a
record's field values are derived from a previous record's `txtRawStateData`.
Records requiring a parent are identified by `txtRawStateData` beginning with:

- `Same` -- variant of the preceding postmark
- `(L)`, `*(L)`, `(L)*` -- latest known use date reference
- `*(E)`, `(E)*` -- earliest known use date reference

Since IDs were assigned in parent-then-child order during import, we iterate
sequentially to establish proper parentage.

In [38]:
PARENT_INDICATOR_PATTERNS = [
    r'^same',
    r'^\(L\)',
    r'^\*\(L\)',
    r'^\(L\)\*',
    r'^\*\(E\)',
    r'^\(E\)\*',
]
COMBINED_INDICATOR_PATTERN = '|'.join(PARENT_INDICATOR_PATTERNS)

def should_have_parent(txt):
    """Check if txtRawStateData indicates this record should have a parent."""
    if pd.isna(txt) or str(txt).strip() == '':
        return False
    return bool(re.match(COMBINED_INDICATOR_PATTERN, str(txt).strip(), re.IGNORECASE))

def has_parent_assigned(row):
    """Check if nRawStateDataID_parent indicates a true parent (not self, not 0, not empty)."""
    parent = row['nRawStateDataID_parent']
    self_id = row['nRawStateDataID']
    if pd.isna(parent) or str(parent).strip() == '' or str(parent) == '0':
        return False
    return str(parent) != str(self_id)

df['_should_have_parent'] = df['txtRawStateData'].apply(should_have_parent)
df['_has_parent_assigned'] = df.apply(has_parent_assigned, axis=1)

print("=== Current Parent-Child Relationship Status ===")
print(f"Records that SHOULD have a parent (text-based): {df['_should_have_parent'].sum():,}")
print(f"Records that DO have a parent assigned: {df['_has_parent_assigned'].sum():,}")

=== Current Parent-Child Relationship Status ===
Records that SHOULD have a parent (text-based): 13,935
Records that DO have a parent assigned: 204


In [39]:
# Identify mismatches
type_a = df[df['_should_have_parent'] & ~df['_has_parent_assigned']]
type_b = df[~df['_should_have_parent'] & df['_has_parent_assigned']]

print(f"Type A (should have parent, doesn't): {len(type_a):,}")
print(f"Type B (has parent, text doesn't indicate): {len(type_b):,}")

Type A (should have parent, doesn't): 13,854
Type B (has parent, text doesn't indicate): 123


<a id="section-5-1"></a>

### 5.1 Type B: Duplicate Records from Botched Import

Type B records have a parent assigned but their catalog text doesn't begin with
any relationship indicator. Investigation shows these are duplicate records from
a failed import affecting Virginia and West Virginia -- all 123 have identical
`txtRawStateData` to their assigned parents.

In [40]:
if len(type_b) > 0:
    # Verify content matches parent
    identical_count = 0
    for _, row in type_b.iterrows():
        parent_id = row['nRawStateDataID_parent']
        child_txt = str(row['txtRawStateData']).strip() if pd.notna(row['txtRawStateData']) else ''
        parent_row = df[df['nRawStateDataID'] == parent_id]
        if len(parent_row) > 0:
            parent_txt = str(parent_row.iloc[0]['txtRawStateData']).strip() if pd.notna(parent_row.iloc[0]['txtRawStateData']) else ''
            if child_txt == parent_txt:
                identical_count += 1

    print(f"Type B records with IDENTICAL txtRawStateData to parent: {identical_count} of {len(type_b)}")
    print(f"\nState distribution:")
    print(type_b['nStateID'].value_counts())
else:
    print("No Type B records found.")

Type B records with IDENTICAL txtRawStateData to parent: 123 of 123

State distribution:
nStateID
46    90
48    33
Name: count, dtype: int64


In [41]:
# Mark Type B records as duplicates
df['_is_duplicate'] = ~df['_should_have_parent'] & df['_has_parent_assigned']
duplicate_count = df['_is_duplicate'].sum()

print(f"Duplicate records identified (Type B): {duplicate_count:,}")

Duplicate records identified (Type B): 123


<a id="section-5-2"></a>

### 5.2 Parent-Child Reassignment

Reassign parentage for Type A records. The algorithm:

1. Sort by `nRawStateDataID` (ascending) to process in import order
2. Track current parent context as we iterate
3. Records starting with a relationship indicator get assigned the current parent
4. Records without an indicator become the new parent context
5. Duplicates (Type B) are skipped

In [42]:
def reassign_parents(dataframe):
    """
    Reassign nRawStateDataID_parent based on txtRawStateData indicators.
    Returns a copy with updated parent assignments and a count of changes.
    """
    df_work = dataframe.copy()
    df_work = df_work.sort_values('nRawStateDataID', key=lambda x: x.astype(int))
    df_work = df_work.reset_index(drop=True)

    changes = 0
    current_parent_id = None
    new_parents = []

    for idx, row in df_work.iterrows():
        record_id = row['nRawStateDataID']
        old_parent = row['nRawStateDataID_parent']
        txt = str(row['txtRawStateData']).strip() if pd.notna(row['txtRawStateData']) else ''
        is_dup = row['_is_duplicate'] if '_is_duplicate' in row else False

        # Skip duplicates - leave their parent as-is
        if is_dup:
            new_parents.append(old_parent)
            continue

        needs_parent = should_have_parent(txt)

        if needs_parent and current_parent_id is not None:
            new_parent = current_parent_id
            if str(old_parent) != str(new_parent):
                changes += 1
            new_parents.append(new_parent)
        else:
            current_parent_id = record_id
            new_parent = record_id
            if str(old_parent) != str(new_parent):
                changes += 1
            new_parents.append(new_parent)

    df_work['nRawStateDataID_parent'] = new_parents
    return df_work, changes

print(f"Records to process: {len(df):,}")
print(f"Duplicates to skip: {duplicate_count:,}")

Records to process: 52,046
Duplicates to skip: 123


In [43]:
# Perform the reassignment
df, parent_changes = reassign_parents(df)
print(f"Total parent values changed: {parent_changes:,}")

Total parent values changed: 19,039


In [44]:
# Verify the reassignment
df['_should_have_parent'] = df['txtRawStateData'].apply(should_have_parent)
df['_has_parent_assigned'] = df.apply(has_parent_assigned, axis=1)

df_non_dup = df[~df['_is_duplicate']]
type_a_after = df_non_dup[df_non_dup['_should_have_parent'] & ~df_non_dup['_has_parent_assigned']]

print(f"Records that SHOULD have parent: {df_non_dup['_should_have_parent'].sum():,}")
print(f"Records that DO have parent: {df_non_dup['_has_parent_assigned'].sum():,}")
print(f"Remaining Type A mismatches: {len(type_a_after):,}")

Records that SHOULD have parent: 13,935
Records that DO have parent: 13,935
Remaining Type A mismatches: 0


In [45]:
# Show sample reassigned relationships
children = df[
    (df['_should_have_parent'])
    & (df['_has_parent_assigned'])
    & (~df['_is_duplicate'])
]

for _, child in children.head(10).iterrows():
    child_id = child['nRawStateDataID']
    parent_id = child['nRawStateDataID_parent']
    child_txt = str(child['txtRawStateData'])[:60] if pd.notna(child['txtRawStateData']) else 'NULL'

    parent = df[df['nRawStateDataID'] == parent_id]
    if len(parent) > 0:
        parent_txt = str(parent.iloc[0]['txtRawStateData'])[:60] if pd.notna(parent.iloc[0]['txtRawStateData']) else 'NULL'
        print(f"Parent {parent_id}: {parent_txt}...")
        print(f"  Child {child_id}: {child_txt}...")
        print()

Parent 1: Alexa.(Alexandria)(E)(May 21, 1772;Ms;Black) 1,500...
  Child 2: (L)(Sept. 15, 1774) 1,000...

Parent 8: FREDERICKSBURG(“F” 5mm high, used as bkstp)(March 1, 1775;SL...
  Child 9: (L)(June 27, 1775) 1,000...

Parent 15: Norf(Norfolk)(E)(Aug. 21, 1765;Ms;Black) 1,500...
  Child 16: (L)(Dec. 6, 1765) 1,200...

Parent 17: Nfk(Norfolk)(Nov. 20, 1772;Ms;Red) 1,000...
  Child 18: Same(--, 1773;Ms;Black) 1,000...

Parent 19: NORFOLK(backstamp)(E)(Feb. 11, 1775;SL-29x5,MDD below;Black)...
  Child 20: (L)(Oct. 2, 1775) 1,200...

Parent 19: NORFOLK(backstamp)(E)(Feb. 11, 1775;SL-29x5,MDD below;Black)...
  Child 21: Same(May 6, 1775;MDD same line) 1,500...

Parent 19: NORFOLK(backstamp)(E)(Feb. 11, 1775;SL-29x5,MDD below;Black)...
  Child 22: Same(July 25, 1775;MDD below;Red) 1,500...

Parent 26: SUFFOLK(April 12, 1775;SL-30x5,MDD below;Red) 1500...
  Child 27: Same(July 25, 1775;Black) 1,200...

Parent 28: WBurg(Williamsburg)(E)(Nov. 14, 1734;Ms;Black) 2,500...
  Child 29: Same(--, 174

---

<a id="section-6"></a>

## 6. Family Text Rollup

After parent-child relationships are established, create a rolled-up view of each
catalog entry family. `txtRawStateDataTemp` gets an uppercase concatenation of the
parent's `txtRawStateData` plus all children's text (sorted by ID).

Standalone records (no children) get their own text uppercased.

In [46]:
def create_family_rollup(dataframe, delimiter=' | '):
    df = dataframe.copy()

    # Clean text from source column
    df['_txt_clean'] = (
        df['txtRawStateData']
          .fillna('')
          .astype(str)
          .replace(['nan', 'None', 'null', 'NaN'], '')
          .str.strip()
    )

    # Build stable numeric IDs
    id_num = pd.to_numeric(df['nRawStateDataID'], errors='coerce')
    parent_num = pd.to_numeric(df['nRawStateDataID_parent'], errors='coerce')

    # Family id: parent id if it exists, otherwise own id
    df['_family_id'] = parent_num.where(parent_num.notna(), id_num)

    # Fallback for rows missing both
    fallback = df.index.to_series().astype('int64')
    df['_family_id'] = df['_family_id'].fillna(fallback)

    # Parent-first ordering: parent row is the one whose ID == family_id
    df['_is_parent'] = (id_num == df['_family_id']).fillna(False)

    df_sorted = df.sort_values(
        by=['_family_id', '_is_parent', 'nRawStateDataID'],
        ascending=[True, False, True]
    )

    # One rollup per family, deduplicated, then broadcast
    def rollup_text(series):
        vals = [v for v in series if v]
        if not vals:
            return ""
        seen = set()
        uniq = []
        for v in vals:
            if v not in seen:
                seen.add(v)
                uniq.append(v)
        return delimiter.join(uniq).upper()

    rollups = (
        df_sorted
        .groupby('_family_id', sort=False)['_txt_clean']
        .apply(rollup_text)
    )

    df['txtRawStateDataTemp'] = df['_family_id'].map(rollups)

    # Metrics
    valid = df['txtRawStateDataTemp'] != ""
    family_count = df.loc[valid, '_family_id'].nunique()
    record_count = int(valid.sum())

    print("Family rollup complete:")
    print(f"  Families processed: {family_count:,}")
    print(f"  Records updated: {record_count:,}")

    return df

In [47]:
# Perform the family text rollup
df = create_family_rollup(df, delimiter='  ')

Family rollup complete:
  Families processed: 36,872
  Records updated: 50,930


In [48]:
# Rollup statistics
temp_populated = df['txtRawStateDataTemp'].notna() & (df['txtRawStateDataTemp'] != '')
print(f"Records with txtRawStateDataTemp populated: {temp_populated.sum():,} ({temp_populated.sum()/len(df)*100:.1f}%)")

family_sizes = df.groupby('nRawStateDataID_parent').size()
print(f"\nFamily size distribution:")
print(f"  Single records (no children): {(family_sizes == 1).sum():,}")
print(f"  2 records (parent + 1 child): {(family_sizes == 2).sum():,}")
print(f"  3+ records: {(family_sizes >= 3).sum():,}")
print(f"  Largest family: {family_sizes.max()} records")

Records with txtRawStateDataTemp populated: 50,930 (97.9%)

Family size distribution:
  Single records (no children): 30,793
  2 records (parent + 1 child): 4,440
  3+ records: 2,755
  Largest family: 82 records


---

<a id="section-7"></a>

## 7. Default / Main Image Synchronization

The explorer notebook found discrepancies between `txtDefaultImage` in `tblRawStateData`
and `ynMainImage` in `tblTownmarkImages`. `txtDefaultImage` is the source of truth.
This section brings `tblTownmarkImages` into alignment, then produces a cleaned
`postmark_image` output with renamed columns and orphaned records dropped.

In [49]:
IMAGES_INPUT_FILE = './wip/in/tblTownmarkImages_orig.csv'
df_images = pd.read_csv(IMAGES_INPUT_FILE, low_memory=False)
print(f"Loaded {len(df_images):,} image records")

# Clean IDs: not garbage and not duplicate
clean_ids = set(df.loc[
    ~df['_is_garbage'] & ~df['_is_duplicate'],
    'nRawStateDataID'
].astype(int))

# Build lookup: nRawStateDataID -> txtDefaultImage (only where populated)
default_image_lookup = (
    df.loc[
        df['txtDefaultImage'].notna() & (df['txtDefaultImage'] != ''),
        ['nRawStateDataID', 'txtDefaultImage']
    ]
    .set_index('nRawStateDataID')['txtDefaultImage']
    .to_dict()
)
# Ensure integer keys
default_image_lookup = {int(k): v for k, v in default_image_lookup.items()}

print(f"Records with txtDefaultImage populated: {len(default_image_lookup):,}")

# Operation A: For each record that has a txtDefaultImage, the image whose
# txtFilename matches should be ynMainImage=1 and all others should be 0.

fixed_flag_count = 0
for idx, row in df_images.iterrows():
    rid = int(row['nRawStateDataID'])
    if rid not in default_image_lookup:
        continue

    expected_default = default_image_lookup[rid]
    is_match = (row['txtFilename'] == expected_default)
    current_flag = int(row['ynMainImage'])

    if is_match and current_flag != 1:
        df_images.at[idx, 'ynMainImage'] = 1
        fixed_flag_count += 1
    elif not is_match and current_flag == 1:
        df_images.at[idx, 'ynMainImage'] = 0
        fixed_flag_count += 1

print(f"\nOperation A: Fixed {fixed_flag_count} ynMainImage flags")

# Operation B: Where txtDefaultImage references a filename not present in any
# related image record, create a new row.

existing_pairs = set(
    zip(df_images['nRawStateDataID'].astype(int), df_images['txtFilename'])
)

new_rows = []
for rid, filename in default_image_lookup.items():
    if (rid, filename) not in existing_pairs:
        new_rows.append({
            'nTownmarkImageID': df_images['nTownmarkImageID'].max() + 1 + len(new_rows),
            'nRawStateDataID': rid,
            'txtFilename': filename,
            'txtView': '',
            'txtDetails': '',
            'ynMainImage': 1,
            'nOrder': 0,
            'ynActive': 1,
            'ynDeleted': 0,
            'dtEntered': pd.Timestamp.now().strftime('%Y-%m-%d'),
            'dtUpdated': pd.Timestamp.now().strftime('%Y-%m-%d'),
            'imageStatus': '',
            'submitterName': '',
            'submitterEmail': '',
            'ynEmailCheck': 0
        })

if new_rows:
    df_images = pd.concat([df_images, pd.DataFrame(new_rows)], ignore_index=True)

print(f"Operation B: Inserted {len(new_rows)} missing image records")
print(f"\nTotal image records after fixes: {len(df_images):,}")

Loaded 11,022 image records
Records with txtDefaultImage populated: 6,548

Operation A: Fixed 212 ynMainImage flags
Operation B: Inserted 1 missing image records

Total image records after fixes: 11,023


In [50]:
IMAGES_OUTPUT_FILE = './wip/out/postmark_image.csv'

# Drop orphaned image records (no matching postmark in clean set)
before_orphan_drop = len(df_images)
df_images['nRawStateDataID'] = df_images['nRawStateDataID'].astype(int)
df_images = df_images[df_images['nRawStateDataID'].isin(clean_ids)].copy()
orphans_dropped = before_orphan_drop - len(df_images)
print(f"Dropped {orphans_dropped:,} orphaned image records (no matching clean postmark)")

# Drop records with no filename
before_filename_drop = len(df_images)
df_images = df_images[df_images['txtFilename'].notna() & (df_images['txtFilename'] != '')].copy()
filename_nulls_dropped = before_filename_drop - len(df_images)
print(f"Dropped {filename_nulls_dropped:,} image records with null/blank filename")

# Rename columns and select final output
postmark_images = df_images.rename(columns={
    'nTownmarkImageID': 'id',
    'nRawStateDataID': 'postmark_id',
    'txtFilename': 'filename',
    'txtView': 'type',
    'ynMainImage': 'is_default',
    'nOrder': 'display_order'
})[['id', 'postmark_id', 'filename', 'type', 'is_default', 'display_order']]

# Ensure clean types
postmark_images['id'] = postmark_images['id'].astype(int)
postmark_images['postmark_id'] = postmark_images['postmark_id'].astype(int)
postmark_images['is_default'] = postmark_images['is_default'].astype(int)
postmark_images['display_order'] = postmark_images['display_order'].astype(int)

print(f"\nFinal postmark_images: {len(postmark_images):,} records")
print(f"Columns: {list(postmark_images.columns)}")
print(f"\nis_default distribution:")
print(postmark_images['is_default'].value_counts().to_string())

# Save
postmark_images.to_csv(IMAGES_OUTPUT_FILE, index=False)
print(f"\nSaved to: {IMAGES_OUTPUT_FILE}")

Dropped 793 orphaned image records (no matching clean postmark)
Dropped 34 image records with null/blank filename

Final postmark_images: 10,196 records
Columns: ['id', 'postmark_id', 'filename', 'type', 'is_default', 'display_order']

is_default distribution:
is_default
1    6316
0    3880

Saved to: ./wip/out/postmark_image.csv


---

<a id="section-8"></a>

## 8. State Lookup Merge

Load the states reference table and merge `txtState` and `txtStateAbv` into the
postmark dataframe via `nStateID`.

In [51]:
STATES_INPUT_FILE = './wip/in/tblStates_orig.csv'
df_states = pd.read_csv(STATES_INPUT_FILE, dtype=str, low_memory=False, keep_default_na=False)

# Keep only the columns we need
df_states = df_states[['nStateID', 'txtState', 'txtStateAbv']].copy()

print(f"Loaded {len(df_states):,} state records")
print(f"\nSample:")
print(df_states.head(10).to_string(index=False))

Loaded 55 state records

Sample:
nStateID    txtState txtStateAbv
       1     Alabama          AL
       2      Alaska          AK
       3     Arizona          AZ
       4    Arkansas          AR
       5  California          CA
       6    Colorado          CO
       7 Connecticut          CT
       8    Delaware          DE
       9     Florida          FL
      10     Georgia          GA


In [52]:
# Merge state info into postmark dataframe
before_cols = len(df.columns)
df = df.merge(df_states, on='nStateID', how='left')

unmatched = df['txtStateAbv'].isna().sum()
print(f"Columns before merge: {before_cols}, after: {len(df.columns)}")
print(f"Records matched to a state: {len(df) - unmatched:,}")
if unmatched > 0:
    print(f"Records with no state match: {unmatched:,}")

print(f"\nState distribution (top 15):")
print(df['txtStateAbv'].value_counts().head(15).to_string())


Columns before merge: 16, after: 18
Records matched to a state: 52,046

State distribution (top 15):
txtStateAbv
CT    8532
NY    5678
MA    3864
PA    2793
OH    2751
VA    2406
AL    2071
ME    1730
IL    1593
MI    1462
WI    1381
VT    1268
IN    1111
NC    1110
NH    1001


---

<a id="section-9"></a>

## 9. Save Results

In [53]:
# Separate records by category
duplicate_records = df[df['_is_duplicate']]

# Clean: not garbage AND not duplicate
clean_mask = ~df['_is_garbage'] & ~df['_is_duplicate']
clean_records = df[clean_mask]

# Garbage: everything else
garbage_records = df[df['_is_garbage']]

# Ensure output directory exists
Path(OUTPUT_FILE).parent.mkdir(parents=True, exist_ok=True)

# Save combined files
clean_records.to_csv(OUTPUT_FILE, index=False)
print(f"Saved {len(clean_records):,} clean records to: {OUTPUT_FILE}")

garbage_records.to_csv(GARBAGE_FILE, index=False)
print(f"Saved {len(garbage_records):,} garbage records to: {GARBAGE_FILE}")

duplicate_records.to_csv(DUPLICATES_FILE, index=False)
print(f"Saved {len(duplicate_records):,} duplicate records to: {DUPLICATES_FILE}")

# Per-state clean record files
output_dir = Path(OUTPUT_FILE).parent
state_groups = clean_records.groupby('txtStateAbv')
state_count = 0

for abv, group in state_groups:
    if pd.isna(abv) or str(abv).strip() == '':
        continue
    filename = f"{str(abv).lower()}-postmark.csv"
    filepath = output_dir / filename
    group.to_csv(filepath, index=False)
    state_count += 1

print(f"\nSaved {state_count} per-state files to: {output_dir}/")
print(f"Example: {output_dir}/ct-postmark.csv")


Saved 50,829 clean records to: ./wip/out/postmark.csv
Saved 1,094 garbage records to: ./wip/out/tblRawStateData_garbage.csv
Saved 123 duplicate records to: ./wip/out/tblRawStateData_duplicates.csv

Saved 53 per-state files to: wip/out/
Example: wip/out/ct-postmark.csv


In [54]:
# Final column inventory
print(f"\nFinal columns ({len(df.columns)}):")
for i, col in enumerate(df.columns):
    print(f"  {i:2}. {col}")


Final columns (18):
   0. nRawStateDataID
   1. nRawStateDataID_parent
   2. nStateID
   3. txtRawStateData
   4. txtDefaultImage
   5. _is_childless
   6. _is_orphaned
   7. _empty_content
   8. _is_garbage
   9. _should_have_parent
  10. _has_parent_assigned
  11. _is_duplicate
  12. _txt_clean
  13. _family_id
  14. _is_parent
  15. txtRawStateDataTemp
  16. txtState
  17. txtStateAbv
